# ST 554 Analysis of BigData - HW10
Name: Yujin Kim

Course: ST 554 (601) Spring 2026 Analysis of Big Data

Assignment: Homework #10

The purpose of this assignment is to understand how Spark Structured Streaming processes real-time data using a pipeline-based workflow. 
This follows the general Structured Streaming process of:
reading a stream, applying transformations, writing the output, and starting the query.

In [157]:
from pyspark.sql import SparkSession

In [158]:
# create spark session
spark = SparkSession.builder.appName("HW10_Streaming").getOrCreate()

# Part I. Creating Streaming Data Using `rate`

In this part, a streaming DataFrame is created using the rate source, which generates rows continuously over time. 
The results are written to an in-memory table, allowing us to query the full output after stopping the stream.

Instruction: Setup a data stream using `"rate"` format.

In [159]:
# read rate stream
rate_df = (spark.readStream.format("rate").option("rowsPerSecond", 1).load())

Instruction: Prior to starting the stream, set up a sequence of actions using appropriate functions from `pyspark.sql.functions` that uses the `rate` data to
* find the square root of the rate 'value'
* find mod 4 of the rate 'value'

In [160]:
from pyspark.sql.functions import col, sqrt

# calculate square root and mod 4
rate_transformed = (rate_df.withColumn("sqrt_value", sqrt(col("value")))
                    .withColumn("mod_4", col("value") % 4))

The stream is written to a memory sink so that it can be queried later using Spark SQL.

Intruction: To output this, create a `writeStream` that writes to 'meory' (`format("memory")`). Give the query a name (`queryName("...")`) and start it!

In [161]:
# Save as memory sink
query1 = (rate_transformed.writeStream.format("memory").queryName("rate_table").outputMode("append").start())

26/04/20 22:23:47 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-deab8577-296a-4d15-a908-0e47ec88f318. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/20 22:23:47 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Instruction: Let it run for about 30 seconds and then stop the query.

In [162]:
import time

time.sleep(30)
query1.stop()

26/04/20 22:24:17 WARN DAGScheduler: Failed to cancel job group b21f1973-6423-4eac-a17e-690356c51619. Cannot find active jobs for it.
26/04/20 22:24:17 WARN DAGScheduler: Failed to cancel job group b21f1973-6423-4eac-a17e-690356c51619. Cannot find active jobs for it.


After running the stream for approximately 30 seconds, the query is stopped and the results are retrieved from memory.

Instruction: Then output the entire table stored in the query name (`spark.sql("select * from you_table_anme").show()`)

In [163]:
#print entire memory table
spark.sql("SELECT * FROM rate_table").show(truncate = False)

+-----------------------+-----+------------------+-----+
|timestamp              |value|sqrt_value        |mod_4|
+-----------------------+-----+------------------+-----+
|2026-04-20 22:23:47.455|0    |0.0               |0    |
|2026-04-20 22:23:48.455|1    |1.0               |1    |
|2026-04-20 22:23:49.455|2    |1.4142135623730951|2    |
|2026-04-20 22:23:50.455|3    |1.7320508075688772|3    |
|2026-04-20 22:23:51.455|4    |2.0               |0    |
|2026-04-20 22:23:52.455|5    |2.23606797749979  |1    |
|2026-04-20 22:23:53.455|6    |2.449489742783178 |2    |
|2026-04-20 22:23:54.455|7    |2.6457513110645907|3    |
|2026-04-20 22:23:55.455|8    |2.8284271247461903|0    |
|2026-04-20 22:23:56.455|9    |3.0               |1    |
|2026-04-20 22:23:57.455|10   |3.1622776601683795|2    |
|2026-04-20 22:23:58.455|11   |3.3166247903554   |3    |
|2026-04-20 22:23:59.455|12   |3.4641016151377544|0    |
|2026-04-20 22:24:00.455|13   |3.605551275463989 |1    |
|2026-04-20 22:24:01.455|14   |

# Part II. Using data from a CSV with a Pipeline

In this part, a preprocessing pipeline is created using a static dataset and then applied to streaming data.
The fitted pipeline is then applied to incoming CSV files in a streaming folder, simulating real-time data processing.

Instruction: There are six `bikeDetails` sub datasets available on the assignment link. The one named `bikeDetails_for_fit.csv`
should be read in as a spark (SQL) data frame. With this spark SQL data frame do the following

In [164]:
from pyspark.ml import Pipeline

#read csv for fit
fit_path = "bikeDetails_for_fit.csv"

In [165]:
bike_fit_df = (spark.read.option("header", True).option("inferSchema", True).csv(fit_path))

In [166]:
bike_fit_df.printSchema()
bike_fit_df.show(5, truncate = False)

root
 |-- name: string (nullable = true)
 |-- selling_price: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- seller_type: string (nullable = true)
 |-- owner: string (nullable = true)
 |-- km_driven: integer (nullable = true)
 |-- ex_showroom_price: integer (nullable = true)

+-----------------------------------+-------------+----+-----------+---------+---------+-----------------+
|name                               |selling_price|year|seller_type|owner    |km_driven|ex_showroom_price|
+-----------------------------------+-------------+----+-----------+---------+---------+-----------------+
|Royal Enfield Classic 350          |175000       |2019|Individual |1st owner|350      |NULL             |
|Honda Dio                          |45000        |2017|Individual |1st owner|5650     |NULL             |
|Royal Enfield Classic Gunmetal Grey|150000       |2018|Individual |1st owner|12000    |148114           |
|Yamaha Fazer FI V 2.0 [2016-2018]  |65000        |2015|Indiv

Instructin: use an SQLTransformer with the following statement (this does some log transforms, renames a variable,
and creates a dummy variable from categorical variable):

`SELECT log(selling_price) as label, year, log(km_driven) as log_km_driven,
CASE WHEN owner = ’1st owner’ THEN 1 ELSE 0 END AS one_owner
FROM __THIS__`

In [167]:
from pyspark.ml.feature import SQLTransformer, VectorAssembler

# Set SQLTransformer
sql_transformer = SQLTransformer(statement = """
SELECT 
    log(selling_price) as label, year, log(km_driven) as log_km_driven,
    CASE WHEN owner = '1st owner' THEN 1 ELSE 0 END AS one_owner
FROM __THIS__
""")

Instruction: use a `VectorAssembler` to create a `features` column. The `features` column should include the `year`,
`log_km_driven`, and `one_owner variables`.

In [168]:
#Set Vector Assembler
assembler  = VectorAssembler(
    inputCols = ["year", "log_km_driven", "one_owner"],
    outputCol = "features")

The pipeline combines multiple transformation steps into a single workflow, ensuring that the same preprocessing is consistently applied to both training and streaming data.

Instruction: create a `Pipeline` with the two steps above (`SQLTransformer` then `VectorAssembler`), fit this pipeline to the `SQL` data frame and save this as an object.

In [169]:
pipeline = Pipeline(stages = [sql_transformer, assembler])   #create a Pipeline
pipeline_model = pipeline.fit(bike_fit_df)                   #fit pipeline to the SQL df

Instruction: Now we want to set up a read stream to look for csv files placed into a folder (the five `bikeDetails_add*.csv` files). When a csv comes in, we want to transform it using the fitted pipeline’s `.transform()` method!

In [170]:
#check fitted pipeline results
pipeline_model.transform(bike_fit_df).select("label", "features").show(10, truncate = False)

+------------------+-------------------------------+
|label             |features                       |
+------------------+-------------------------------+
|12.072541252905651|[2019.0,5.857933154483459,1.0] |
|10.714417768752456|[2017.0,8.639410824140487,1.0] |
|11.918390573078392|[2018.0,9.392661928770137,1.0] |
|11.082142548877775|[2015.0,10.043249494911286,1.0]|
|9.903487552536127 |[2011.0,9.95227771670556,0.0]  |
|9.798127036878302 |[2010.0,11.002099841204238,1.0]|
|11.2708539037705  |[2018.0,9.740968623038354,1.0] |
|12.100712129872347|[2008.0,10.571316925111784,0.0]|
|10.308952660644293|[2010.0,10.373491181781864,1.0]|
|10.819778284410283|[2016.0,10.645424897265505,1.0]|
+------------------+-------------------------------+
only showing top 10 rows


In [171]:
# prepare streaming folder
stream_dir = "bike_stream_input"

In [172]:
import os

if os.path.exists(stream_dir):
    shutil.rmtree(stream_dir)
os.makedirs(stream_dir)

A streaming DataFrame is created to monitor a directory for incoming CSV files. 
Each file is processed as it arrives.

Instruction: You’ll need a schema to set up the `readStream`. You can use the SQL data frame’s schema from above!
(`.schema` attribute)

In [173]:
bike_schema = bike_fit_df.schema #set schema
bike_stream_df = (spark.readStream.schema(bike_schema).option("header", True).csv(stream_dir)) #set readStream

The fitted pipeline is applied to the streaming data using the transform method, ensuring consistency with the preprocessing applied to the training data.

In [174]:
#apply fitted pipeline
bike_stream_transformed = pipeline_model.transform(bike_stream_df)

In [175]:
#print console sink
query2 = (bike_stream_transformed.writeStream
          .format("console")
          .outputMode("append")
          .option("truncate", False)
          .start())

26/04/20 22:24:18 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-3e4372cc-825e-4558-9e6d-19acf6651198. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/20 22:24:18 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Instruction: Each file you’ll be adding to the folder has a header!

In [176]:
source_files = ["bikeDetails_add1.csv", 
                "bikeDetails_add2.csv", 
                "bikeDetails_add3.csv", 
                "bikeDetails_add4.csv",
                "bikeDetails_add5.csv"]

In [177]:
import shutil

for f in source_files:
    dest = os.path.join(stream_dir, os.path.basename(f))
    shutil.copy(f, dest)
    print(f"Added: {os.path.basename(f)}")
    time.sleep(5)

Added: bikeDetails_add1.csv
-------------------------------------------
Batch: 0
-------------------------------------------
+------------------+----+------------------+---------+-------------------------------+
|label             |year|log_km_driven     |one_owner|features                       |
+------------------+----+------------------+---------+-------------------------------+
|8.987196820661973 |2003|10.887436932884098|1        |[2003.0,10.887436932884098,1.0]|
|11.156250521031495|2018|9.615805480084347 |1        |[2018.0,9.615805480084347,1.0] |
|10.819778284410283|2016|8.987196820661973 |1        |[2016.0,8.987196820661973,1.0] |
|10.46310334047155 |2015|10.582738627903963|1        |[2015.0,10.582738627903963,1.0]|
|9.903487552536127 |2006|11.225243392518447|1        |[2006.0,11.225243392518447,1.0]|
|10.819778284410283|2012|10.239959789157341|1        |[2012.0,10.239959789157341,1.0]|
|10.51867319162636 |2008|11.03488966402723 |1        |[2008.0,11.03488966402723,1.0] |
|11.1

The results are written to the console in append mode, allowing us to observe how each incoming file is processed in real time.

In [178]:
query2.stop()

26/04/20 22:24:43 WARN DAGScheduler: Failed to cancel job group 91545d9c-ca73-4e82-a9bf-d1fc0bab9e72. Cannot find active jobs for it.
26/04/20 22:24:43 WARN DAGScheduler: Failed to cancel job group 91545d9c-ca73-4e82-a9bf-d1fc0bab9e72. Cannot find active jobs for it.


This assignment demonstrates how Spark Structured Streaming can be used to process data in real time using a structured pipeline approach.

In Part 1, we verified how streaming data can be generated and transformed continuously, and how intermediate results can be stored and queried.

In Part 2, we showed how a preprocessing pipeline fitted on static data can be applied to streaming data. This highlights the ability to integrate machine learning workflows with real-time data processing.